In [1]:
# Import Section
import pandas as pd
import warnings
import logging
from utils.get_ticker import *
from utils.load_data import *
from fundamental_feature_engineering import *

logging.basicConfig(level=logging.INFO)
warnings.filterwarnings('ignore')

# Dataframe Init
original_df = pd.read_csv('/root/vynixmodelling/dataset/TSLA_original.csv')

# Logging Init
# logging location: /root/vynixmodelling/ML_RL/logs
# log name format: {log_name}_{time}.log
logging.basicConfig(filename='/root/vynixmodelling/ML_RL/logs/main.log', level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Preprocessing
df = original_df.copy()
logging.info(f"Original dataframe shape: {df.shape}")

# Exclude rows with NaN or null values
non_null_df = df.dropna()

# Print total rows in the dataframe without NaN or null values
logging.info(f"Total rows in df without NaN or null values: {len(non_null_df)}")

# Convert 'time' column to datetime format and print the first few rows
non_null_df['time_converted'] = pd.to_datetime(non_null_df['time'], unit='s').dt.strftime('%d%m%Y')
# print(non_null_df[['time', 'time_converted']].head(1))
print(non_null_df.head)
print(non_null_df.tail)
print(non_null_df.info())
print(non_null_df.describe())
print(non_null_df.columns)

# Data Gathering with Fundamental data
# 1. Process Fundamental Data from Local Files
# Menggunakan fungsi baru untuk membaca data lokal berdasarkan CIK
pivoted_df = process_fundamental_data_local("TSLA")

# Jika gagal memuat dari lokal, fallback ke API
if pivoted_df is None:
    print("Fallback to API method...")
    fundamental_data = get_fundamental_data("TSLA")
    pivoted_df = process_fundamental_data(fundamental_data, "TSLA")
    print("Data loaded from API successfully")
else:
    # Baca kembali CSV yang sudah diproses untuk mendapatkan DataFrame
    pivoted_df = pd.read_csv('/root/vynixmodelling/dataset/data_fundamental/TSLA_time.csv', index_col=0)
    print(f"Data loaded from local files successfully: {pivoted_df.shape}")

# 3. Fundamental Data Financial Feature Engineering
from fundamental_feature_engineering import apply_feature_engineering
enhanced_fundamental_df = apply_feature_engineering(pivoted_df, "TSLA")
# print(enhanced_fundamental_df.head(1))

# 4. Teknikal and Fundamental Data Preprocessing
from technical_fundamental_preprocessing import preprocess_technical_fundamental_data

# Process data using complete pipeline
logging.info("Processing technical and fundamental data...")
output_path = '/root/vynixmodelling/dataset/main_processed_data.csv'

filtered_df = preprocess_technical_fundamental_data(
    technical_df=non_null_df,
    fundamental_df=enhanced_fundamental_df,
    output_path=output_path,
    start_period='2012-Q2',
    end_period='2025-Q2'
)

logging.info(f"Data processing completed: {filtered_df.shape}")
logging.info(f"Processed data saved to: {output_path}")

# Display final results
print(f"\nFinal processed data shape: {filtered_df.shape}")
print(f"Date range: {filtered_df['date'].min()} to {filtered_df['date'].max()}")
print(f"Technical columns: {len([col for col in filtered_df.columns if any(tech in col.lower() for tech in ['open', 'high', 'low', 'close', 'volume'])])}")
print(f"Fundamental columns: {len([col for col in filtered_df.columns if col not in ['time', 'date', 'datetime'] and not any(tech in col.lower() for tech in ['open', 'high', 'low', 'close', 'volume'])])}")
print("\nPreprocessing completed successfully!")
print(f'Data completeness: {(filtered_df.count().sum() / (len(filtered_df) * len(filtered_df.columns)) * 100):.2f}%')

# 5. Implementasi Labelling Triple Barrier Method ke filtered_df
# Semua parameter di init di file ini. main.py.
from triple_barrier.triplebarrier import apply_triple_barrier_labeling
from triple_barrier.visualizebarrier import generate_triple_barrier_visualizations

# Triple Barrier Method Parameters - dapat dikonfigurasi sesuai kebutuhan
TRIPLE_BARRIER_PARAMS = {
    'volatility_window': 20,           # Window untuk menghitung volatilitas
    'upper_barrier_multiplier': 1.0,   # Multiplier untuk upper barrier
    'lower_barrier_multiplier': 1.0,   # Multiplier untuk lower barrier
    'time_barrier_days': 5,            # Maksimum periode untuk menunggu barrier touch
    'verbose': True                    # Tampilkan statistik hasil
}

# Visualization Parameters
VISUALIZATION_PARAMS = {
    'output_dir': '/root/vynixmodelling/ML_RL/logs/visualization',
    'window_size': 50,                 # Jumlah data sebelum dan sesudah untuk visualisasi
    'save_html': True,                 # Simpan dalam format HTML
    'save_png': False,                  # Simpan dalam format PNG (akan menginstal kaleido otomatis)
    'verbose': True                    # Tampilkan log proses
}

print("\n=== Applying Triple Barrier Method ===")
logging.info("Starting Triple Barrier Method labeling...")

# Aplikasikan Triple Barrier Method
triple_barrier_df = apply_triple_barrier_labeling(
    data=filtered_df,
    **TRIPLE_BARRIER_PARAMS
)

# Simpan hasil Triple Barrier ke CSV
triple_barrier_output_path = '/root/vynixmodelling/ML_RL/triple_barrier_results.csv'
triple_barrier_df.to_csv(triple_barrier_output_path, index=False)

# Generate visualisasi Triple Barrier
print("\n=== Generating Triple Barrier Visualizations ===")
logging.info("Generating Triple Barrier visualizations...")

visualization_files = generate_triple_barrier_visualizations(
    data=filtered_df,
    triple_barrier_df=triple_barrier_df,
    **VISUALIZATION_PARAMS
)

logging.info(f"Visualizations generated: {len(visualization_files)} files")
for file_type, file_path in visualization_files.items():
    logging.info(f"{file_type}: {file_path}")

print(f"\n=== Triple Barrier Implementation Complete ===")
print(f"Labels generated: {len(triple_barrier_df)} samples")
print(f"Results saved to: {triple_barrier_output_path}")
# print(f"Visualizations saved to: {VISUALIZATION_PARAMS['output_dir']}")
# print(f"Total visualization files: {len(visualization_files)}")

# Summary statistik final
# 6. Merge Triple Barrier labels dengan filtered_df berdasarkan date
# merged_label_df = triple_barrier_df.copy() + filtered_df.copy(), gunakan triple_barrier_df sebagai
# data utama yang sudah ber-label, lalu tarik data filtered_df sesuai dengan 'date' dari 'decision_date' triple_barrier_df

print("\n=== Merging Triple Barrier Labels with Features ===")
logging.info("Merging triple_barrier_df with filtered_df based on decision_date...")

# Konversi decision_date ke datetime jika belum
triple_barrier_df['decision_date'] = pd.to_datetime(triple_barrier_df['decision_date'])
filtered_df['date'] = pd.to_datetime(filtered_df['date'])

# Merge berdasarkan decision_date dari triple_barrier_df dengan date dari filtered_df
merged_label_df = triple_barrier_df.merge(
    filtered_df, 
    left_on='decision_date', 
    right_on='date', 
    how='left'  # Gunakan left join untuk mempertahankan semua data triple_barrier_df
)

# Hapus kolom duplikat 'date' karena sudah ada 'decision_date'
if 'date' in merged_label_df.columns:
    merged_label_df = merged_label_df.drop('date', axis=1)

print(f"Triple Barrier data shape: {triple_barrier_df.shape}")
print(f"Filtered data shape: {filtered_df.shape}")
print(f"Merged data shape: {merged_label_df.shape}")
print(f"Successfully merged: {len(merged_label_df[merged_label_df.notna().all(axis=1)])} complete rows")

# Simpan hasil merge
merged_output_path = '/root/vynixmodelling/ML_RL/merged_labeled_data.csv'
merged_label_df.to_csv(merged_output_path, index=False)
logging.info(f"Merged labeled data saved to: {merged_output_path}")
print(f"Merged labeled data saved to: {merged_output_path}")

print(f"\n=== Final Summary ===")
print(f"Original processed data shape: {filtered_df.shape}")
print(f"Triple Barrier labels: {len(triple_barrier_df)} samples")
print(f"Merged labeled data shape: {merged_label_df.shape}")
print(f"Date range: {filtered_df['date'].min()} to {filtered_df['date'].max()}")
print(f"Date range: {triple_barrier_df['decision_date'].min()} to {triple_barrier_df['decision_date'].max()}")
print(f"Technical + Fundamental features: {len(filtered_df.columns)} columns")
print(f"Total features in merged data: {len(merged_label_df.columns)} columns")
print(f"Triple Barrier parameters used: {TRIPLE_BARRIER_PARAMS}")
print("\nAll preprocessing, labeling, and merging completed successfully!")

# 6. Training Data Preparation.
# Dataframe yang digunakan: merged_label_df. kolom yang perlu di drop:
# 'time','decision_date', 'upper_barrier', 'lower_barrier', 'time_barrier'


INFO:root:Original dataframe shape: (3816, 14)
INFO:root:Total rows in df without NaN or null values: 3783
INFO:root:Processing technical and fundamental data...
INFO:technical_fundamental_preprocessing:Starting complete preprocessing pipeline
INFO:technical_fundamental_preprocessing:Preparing technical data
INFO:technical_fundamental_preprocessing:Technical data prepared: 3783 rows, 17 columns
INFO:technical_fundamental_preprocessing:Preparing fundamental data
INFO:technical_fundamental_preprocessing:Fundamental data prepared: 53 quarters, 151 features
INFO:technical_fundamental_preprocessing:Converting quarterly data to daily format


<bound method NDFrame.head of             time        open        high         low       close      Volume  \
33    1281965400    1.229999    1.253332    1.217439    1.251999   7293172.0   
34    1282051800    1.263999    1.293331    1.251999    1.276665   6718221.0   
35    1282138200    1.305999    1.305999    1.239999    1.251332   9019989.0   
36    1282224600    1.235998    1.283332    1.221999    1.252665   8686238.0   
37    1282311000    1.243331    1.273999    1.234005    1.273333   4462489.0   
...          ...         ...         ...         ...         ...         ...   
3811  1755869400  321.660000  340.250000  319.690000  340.010000  94016347.0   
3812  1756128600  338.900000  349.530000  335.030000  346.600000  86670037.0   
3813  1756215000  344.930000  351.900000  343.720000  351.670000  76651550.0   
3814  1756301400  351.940000  355.390000  349.156000  349.600000  65519012.0   
3815  1756387800  350.910000  353.550000  340.260000  345.980000  67903224.0   

      His

INFO:technical_fundamental_preprocessing:Daily conversion completed: 4839 days, 151 features
INFO:technical_fundamental_preprocessing:Combining technical and fundamental data
INFO:technical_fundamental_preprocessing:Data combination completed: 3330 rows, 168 columns
INFO:technical_fundamental_preprocessing:Date range: 2012-04-02 to 2025-06-30
INFO:technical_fundamental_preprocessing:Filtering data from 2012-Q2 to 2025-Q2
INFO:technical_fundamental_preprocessing:Filtering completed: 3330 rows remaining
INFO:technical_fundamental_preprocessing:Saving processed data to: /root/vynixmodelling/dataset/main_processed_data.csv
INFO:technical_fundamental_preprocessing:Data saved successfully: 3330 rows, 168 columns
INFO:technical_fundamental_preprocessing:Complete preprocessing pipeline finished successfully
INFO:root:Data processing completed: (3330, 168)
INFO:root:Processed data saved to: /root/vynixmodelling/dataset/main_processed_data.csv



Final processed data shape: (3330, 168)
Date range: 2012-04-02 00:00:00 to 2025-06-30 00:00:00
Technical columns: 8
Fundamental columns: 157

Preprocessing completed successfully!
Data completeness: 100.00%


INFO:root:Starting Triple Barrier Method labeling...


...


=== Triple Barrier Method Results ===
Total samples generated: 3324

First few rows:
  decision_date entry_date   end_date  entry_price  end_price    return  \
0    2012-04-02 2012-04-03 2012-04-03     2.438664   2.533997  0.039092   
1    2012-04-03 2012-04-04 2012-04-04     2.533997   2.333331 -0.079190   
2    2012-04-04 2012-04-05 2012-04-09     2.333331   2.209997 -0.052857   
3    2012-04-05 2012-04-09 2012-04-09     2.298664   2.209997 -0.038573   
4    2012-04-09 2012-04-10 2012-04-10     2.209997   2.163999 -0.020814   

   upper_barrier  lower_barrier barrier_touched  value_at_barrier_touched  \
0       2.507050       2.370278           upper                  2.507050   
1       2.605056       2.462938           lower                  2.462938   
2       2.398763       2.267899           lower                  2.267899   
3       2.363124       2.234204           lower                  2.234204   
4       2.271970       2.148024           lower                  2.148024   



INFO:root:Visualizations generated: 3 files
INFO:root:label_1_html: /root/vynixmodelling/ML_RL/logs/visualization/visualisasi_20250912_192306_label_positif.html
INFO:root:label_-1_html: /root/vynixmodelling/ML_RL/logs/visualization/visualisasi_20250912_192306_label_negatif.html
INFO:root:label_0_html: /root/vynixmodelling/ML_RL/logs/visualization/visualisasi_20250912_192306_label_netral.html
INFO:root:Merging triple_barrier_df with filtered_df based on decision_date...


Saved HTML: /root/vynixmodelling/ML_RL/logs/visualization/visualisasi_20250912_192306_label_positif.html
Saved HTML: /root/vynixmodelling/ML_RL/logs/visualization/visualisasi_20250912_192306_label_negatif.html
Saved HTML: /root/vynixmodelling/ML_RL/logs/visualization/visualisasi_20250912_192306_label_netral.html

=== Triple Barrier Implementation Complete ===
Labels generated: 3324 samples
Results saved to: /root/vynixmodelling/ML_RL/triple_barrier_results.csv

=== Merging Triple Barrier Labels with Features ===
Triple Barrier data shape: (3324, 11)
Filtered data shape: (3330, 168)
Merged data shape: (3324, 178)
Successfully merged: 3324 complete rows


INFO:root:Merged labeled data saved to: /root/vynixmodelling/ML_RL/merged_labeled_data.csv


Merged labeled data saved to: /root/vynixmodelling/ML_RL/merged_labeled_data.csv

=== Final Summary ===
Original processed data shape: (3330, 168)
Triple Barrier labels: 3324 samples
Merged labeled data shape: (3324, 178)
Date range: 2012-04-02 00:00:00 to 2025-06-30 00:00:00
Date range: 2012-04-02 00:00:00 to 2025-06-20 00:00:00
Technical + Fundamental features: 168 columns
Total features in merged data: 178 columns
Triple Barrier parameters used: {'volatility_window': 20, 'upper_barrier_multiplier': 1.0, 'lower_barrier_multiplier': 1.0, 'time_barrier_days': 5, 'verbose': True}

All preprocessing, labeling, and merging completed successfully!


In [2]:
columns_list = merged_label_df.columns.tolist()
for i, col in enumerate(columns_list):
    print(f"'{col}', ", end='')
    if (i + 1) % 5 == 0:  # Cetak 5 kolom per baris
        print()
print() # Tambahkan baris baru di akhir

'decision_date', 'entry_date', 'end_date', 'entry_price', 'end_price', 
'return', 'upper_barrier', 'lower_barrier', 'barrier_touched', 'value_at_barrier_touched', 
'label', 'time', 'open', 'high', 'low', 
'close', 'Volume', 'Histogram', 'MACD', 'Signal', 
'K', 'D', 'Turnover (Cr)', '10 MA Turnover', 'Turnover / 10MA (X)', 
'time_converted', 'datetime', 'AccountsPayableCurrent', 'AccountsReceivableNetCurrent', 'AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment', 
'AccumulatedOtherComprehensiveIncomeLossNetOfTax', 'AdditionalPaidInCapitalCommonStock', 'AllocatedShareBasedCompensationExpense', 'Assets', 'AssetsCurrent', 
'CashAndCashEquivalentsAtCarryingValue', 'CommonStockValue', 'ComprehensiveIncomeNetOfTax', 'CostOfRevenue', 'EmployeeRelatedLiabilitiesCurrent', 
'GrossProfit', 'IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest', 'IncomeTaxExpenseBenefit', 'IncreaseDecreaseInAccountsPayableAndAccruedLiabilities', 'Increase

In [3]:
# Drop unnecessary columns for training
columns_to_drop = [
    'decision_date', 'entry_date', 'end_date', 'end_price',
    'return', 'barrier_touched', 'value_at_barrier_touched',
    'time_converted', 'datetime', 'time', 'time_barrier'
]

# Filter out columns that do not exist in the DataFrame
existing_columns_to_drop = [col for col in columns_to_drop if col in merged_label_df.columns]

if existing_columns_to_drop:
    training_df = merged_label_df.drop(columns=existing_columns_to_drop)
    logging.info(f"Dropped columns: {existing_columns_to_drop}")
else:
    training_df = merged_label_df.copy()
    logging.info("No specified columns to drop were found in the DataFrame.")

logging.info(f"Training DataFrame shape after dropping columns: {training_df.shape}")
print(f"Training DataFrame shape after dropping columns: {training_df.shape}")
print(f"Training DataFrame columns: {training_df.columns.tolist()}")

INFO:root:Dropped columns: ['decision_date', 'entry_date', 'end_date', 'end_price', 'return', 'barrier_touched', 'value_at_barrier_touched', 'time_converted', 'datetime', 'time']
INFO:root:Training DataFrame shape after dropping columns: (3324, 168)


Training DataFrame shape after dropping columns: (3324, 168)
Training DataFrame columns: ['entry_price', 'upper_barrier', 'lower_barrier', 'label', 'open', 'high', 'low', 'close', 'Volume', 'Histogram', 'MACD', 'Signal', 'K', 'D', 'Turnover (Cr)', '10 MA Turnover', 'Turnover / 10MA (X)', 'AccountsPayableCurrent', 'AccountsReceivableNetCurrent', 'AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment', 'AccumulatedOtherComprehensiveIncomeLossNetOfTax', 'AdditionalPaidInCapitalCommonStock', 'AllocatedShareBasedCompensationExpense', 'Assets', 'AssetsCurrent', 'CashAndCashEquivalentsAtCarryingValue', 'CommonStockValue', 'ComprehensiveIncomeNetOfTax', 'CostOfRevenue', 'EmployeeRelatedLiabilitiesCurrent', 'GrossProfit', 'IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest', 'IncomeTaxExpenseBenefit', 'IncreaseDecreaseInAccountsPayableAndAccruedLiabilities', 'IncreaseDecreaseInAccountsReceivable', 'IncreaseDecreaseInOtherNoncurrentLiab